# FASE 2 — ETL: Extracción, Transformación y Carga
## Medicamentos Vitales No Disponibles — INVIMA (2018–2026)

**Proyecto:** Entre la necesidad y la disponibilidad  
**Autores:** Julian David Medina Ceballos | Laura Catalina Mariaca Varona  
**Institución:** Unicomfacauca — Diplomado en Ingeniería y Ciencia de Datos Aplicada  
**Sprint Agile Data:** 1 — Preparación del dato limpio

---

### Estructura de la limpieza y transformacion de dataset

| Paso | Proceso | Qué hace |
|------|---------|----------|
| **E** | Extracción | Cargar el archivo `.xlsx` crudo |
| **T1** | Estandarizar columnas y filas | Renombrar encabezados + limpiar TODOS los valores de texto |
| **T2** | Convertir tipos de dato | Fecha real, numérico, validar categorías |
| **T3** | Tratar valores faltantes y duplicados| NaN + eliminar columnas con >70% o mas vacías asi como datos duplicados |
| **T4** | Normalizar todos los tipos de dato | Texto, fechas, números y categorías |
| **T5** | Crear variables derivadas | Año, mes, trimestre, capítulo CIE-10, indicadores |
| **T6** | Validación final | Verificar calidad antes de ingesta en bd |
| **L** | Carga | Exportar CSV limpio  |



---
## CONFIGURACIÓN Y LIBRERÍAS

In [113]:
%pip install pandas openpyxl --quiet
import sys
import pandas as pd
import numpy as np
import unicodedata
import re
import warnings
warnings.filterwarnings('ignore')

RUTA_CRUDO  = '../datasets/MEDICAMENTOS_VITALES_NO_DISPONIBLES_20260425.xlsx'
RUTA_LIMPIO = '../datasets/mvnd_limpio.csv'

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)

print(f' pandas {pd.__version__} | numpy {np.__version__}')

Note: you may need to restart the kernel to use updated packages.
 pandas 2.3.3 | numpy 2.3.5



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---
##  E — EXTRACCIÓN
Cargamos el archivo crudo y guardamos una copia intacta para comparar antes vs. después de la limpieza y transformacion.

In [114]:
# Carga del archivo original
df_crudo = pd.read_excel(RUTA_CRUDO, sheet_name='Data')
df = df_crudo.copy()  # df es el que transformamos; df_crudo nunca se toca

print(f' Archivo cargado exitosamente')
print(f'   Filas    : {df.shape[0]:,}')
print(f'   Columnas : {df.shape[1]}')
print(f'   Columnas : {list(df.columns)}')

 Archivo cargado exitosamente
   Filas    : 10,017
   Columnas : 16
   Columnas : ['FECHA_DE_AUTORIZACIÓN', 'TIPO_DE_SOLICITUD', 'SOLICITANTE/IMPORTADOR', 'IUM', 'PRINCIPIO_ACTIVO1', 'CONCENTRACIÓN_DELMEDICAMENTO1', 'UNIDAD_MEDIDA1', 'PRINCIPIO_ACTIVO2', 'CONCENTRACIÓN_DEL_MEDICAMENTO2', 'UNIDAD_MEDIDA2', 'FORMA_FARMACÉUTICA', 'NOMBRE_COMERCIAL_', 'CANTIDAD_SOLICITADA', 'PRESENTACIÓN_COMERCIAL', 'DIAGNOSTICO_CIE-1NO REPORTA', 'CÓDIGO_DIAGNOSTICO_CIE-10']


---
## T1 — ESTANDARIZAR COLUMNAS Y FILAS

La estandarización tiene **dos niveles**:

- **T1.1 — Encabezados:** renombrar las columnas con nombres cortos, sin caracteres especiales ni tildes, para escribir código sin errores.
- **T1.2 — Valores de texto en las filas:** aplicar una limpieza base a TODO el contenido de texto del dataset: quitar espacios al inicio y al final, eliminar espacios dobles y convertir a mayúsculas.

> Esta limpieza base es el primer nivel. La normalización profunda (tildes, abreviaturas, puntos) va en T4.

In [115]:
# ── T1.1  Renombrar encabezados de columnas ────────────────────────────────
RENOMBRAR = {
    'FECHA_DE_AUTORIZACIÓN'           : 'FECHA_AUTORIZACION',
    'TIPO_DE_SOLICITUD'               : 'TIPO_SOLICITUD',
    'SOLICITANTE/IMPORTADOR'          : 'IMPORTADOR',
    'IUM'                             : 'IUM',
    'PRINCIPIO_ACTIVO1'              : 'PRINCIPIO_ACTIVO',
    'CONCENTRACIÓN_DELMEDICAMENTO1'   : 'CONCENTRACION',
    'UNIDAD_MEDIDA1'                  : 'UNIDAD_MEDIDA',
    'PRINCIPIO_ACTIVO2'              : 'PRINCIPIO_ACTIVO2',
    'CONCENTRACIÓN_DEL_MEDICAMENTO2'  : 'CONCENTRACION2',
    'UNIDAD_MEDIDA2'                  : 'UNIDAD_MEDIDA2',
    'FORMA_FARMACÉUTICA'             : 'FORMA_FARMACEUTICA',
    'NOMBRE_COMERCIAL_'              : 'NOMBRE_COMERCIAL',
    'CANTIDAD_SOLICITADA'             : 'CANTIDAD',
    'PRESENTACIÓN_COMERCIAL'         : 'PRESENTACION_COMERCIAL',
    'DIAGNOSTICO_CIE-1NO REPORTA'    : 'DIAGNOSTICO',
    'CÓDIGO_DIAGNOSTICO_CIE-10'      : 'CODIGO_CIE10'
}

df.columns = df.columns.str.strip()   # Quitar espacios en los nombres actuales
df = df.rename(columns=RENOMBRAR)

print(' T1.1 — Encabezados renombrados:')
for orig, nuevo in RENOMBRAR.items():
    print(f'   {orig:<45} → {nuevo}')

 T1.1 — Encabezados renombrados:
   FECHA_DE_AUTORIZACIÓN                         → FECHA_AUTORIZACION
   TIPO_DE_SOLICITUD                             → TIPO_SOLICITUD
   SOLICITANTE/IMPORTADOR                        → IMPORTADOR
   IUM                                           → IUM
   PRINCIPIO_ACTIVO1                             → PRINCIPIO_ACTIVO
   CONCENTRACIÓN_DELMEDICAMENTO1                 → CONCENTRACION
   UNIDAD_MEDIDA1                                → UNIDAD_MEDIDA
   PRINCIPIO_ACTIVO2                             → PRINCIPIO_ACTIVO2
   CONCENTRACIÓN_DEL_MEDICAMENTO2                → CONCENTRACION2
   UNIDAD_MEDIDA2                                → UNIDAD_MEDIDA2
   FORMA_FARMACÉUTICA                            → FORMA_FARMACEUTICA
   NOMBRE_COMERCIAL_                             → NOMBRE_COMERCIAL
   CANTIDAD_SOLICITADA                           → CANTIDAD
   PRESENTACIÓN_COMERCIAL                        → PRESENTACION_COMERCIAL
   DIAGNOSTICO_CIE-1NO REPORTA             

In [116]:
# ── T1.2  Limpieza base de TODOS los valores de texto en las filas ─────────
# Aplica a todas las columnas de tipo texto (object):
#   - Quitar espacios al inicio y al final
#   - Eliminar espacios dobles internos
#   - Convertir a MAYUSCULAS

cols_texto = df.select_dtypes(include='object').columns.tolist()

for col in cols_texto:
    df[col] = (
        df[col]
        .astype(str)                        # Convertir todo a string primero
        .str.strip()                         # Quitar espacios al inicio y final
        .str.upper()                         # Todo en mayusculas
        .str.replace(r'\s+', ' ', regex=True)  # Eliminar espacios dobles
        .replace('NAN', np.nan)              # Reconvertir NaN que pandas leyó como texto
    )

print(f' T1.2 — Limpieza base aplicada a {len(cols_texto)} columnas de texto:')
for c in cols_texto:
    print(f'   → {c}')

# ── T1.3  Quitar tildes en TIPO_SOLICITUD ───────────────────────────────────
# El archivo INVIMA trae "MÁS DE UN PACIENTE" y "URGENCIA CLÍNICA".
# Sin este paso, T2.3 falla la validación y ES_URGENCIA queda en 0.

def quitar_tildes(texto):
    if pd.isna(texto):
        return np.nan
    texto = unicodedata.normalize('NFD', str(texto))
    return ''.join(c for c in texto if unicodedata.category(c) != 'Mn')

df['TIPO_SOLICITUD'] = df['TIPO_SOLICITUD'].apply(quitar_tildes)

print('\n T1.3 — Tildes eliminadas en TIPO_SOLICITUD')
print('   Valores únicos:', sorted(df['TIPO_SOLICITUD'].dropna().unique()))

# Verificar ejemplo
print('\nEjemplo IMPORTADOR (primeras 5 filas):')
print(df['IMPORTADOR'].head())

 T1.2 — Limpieza base aplicada a 14 columnas de texto:
   → TIPO_SOLICITUD
   → IMPORTADOR
   → IUM
   → PRINCIPIO_ACTIVO
   → CONCENTRACION
   → UNIDAD_MEDIDA
   → PRINCIPIO_ACTIVO2
   → CONCENTRACION2
   → UNIDAD_MEDIDA2
   → FORMA_FARMACEUTICA
   → NOMBRE_COMERCIAL
   → PRESENTACION_COMERCIAL
   → DIAGNOSTICO
   → CODIGO_CIE10

 T1.3 — Tildes eliminadas en TIPO_SOLICITUD
   Valores únicos: ['MAS DE UN PACIENTE', 'PACIENTE ESPECIFICO', 'URGENCIA CLINICA']

Ejemplo IMPORTADOR (primeras 5 filas):
0           VALENTECH PHARMA COLOMBIA SAS
1              HB HUMAN BIOSCIENCE S.A.S.
2                       GESTIFARMA S.A.S.
3    GLOBAL SERVICE PHARMACEUTICAL S.A.S.
4    GLOBAL SERVICE PHARMACEUTICAL S.A.S.
Name: IMPORTADOR, dtype: object


---
## T2 — CONVERTIR TIPOS DE DATO

Cada columna debe tener el tipo de dato correcto para que Python pueda operar con ella.
- **Fecha:** convertir a `datetime` y quedarnos solo con la fecha (sin la hora `05:00:00` que trae el archivo)
- **Cantidad:** convertir de texto a número entero
- **Categorías:** verificar que `TIPO_SOLICITUD` tenga exactamente los 3 valores válidos del reglamento

In [117]:
# ── T2.1  FECHA → datetime (solo fecha, sin hora) ─────────────────────────
df['FECHA_AUTORIZACION'] = pd.to_datetime(
    df['FECHA_AUTORIZACION'], errors='coerce'
).dt.normalize()   # .normalize() deja solo la fecha, elimina la hora 05:00:00

fechas_invalidas = df['FECHA_AUTORIZACION'].isna().sum()

print('T2.1 — FECHA_AUTORIZACION convertida a datetime')
print(f'   Tipo         : {df["FECHA_AUTORIZACION"].dtype}')
print(f'   Fecha mínima : {df["FECHA_AUTORIZACION"].min().date()}')
print(f'   Fecha máxima : {df["FECHA_AUTORIZACION"].max().date()}')
print(f'   Fechas inválidas (no convertibles): {fechas_invalidas}')
print(f'\n   Ejemplo:')
print(df['FECHA_AUTORIZACION'].head(3))

T2.1 — FECHA_AUTORIZACION convertida a datetime
   Tipo         : datetime64[ns]
   Fecha mínima : 2018-01-09
   Fecha máxima : 2026-06-12
   Fechas inválidas (no convertibles): 0

   Ejemplo:
0   2026-06-12
1   2026-03-19
2   2026-03-19
Name: FECHA_AUTORIZACION, dtype: datetime64[ns]


In [93]:
# ── T2.2  CANTIDAD → numérico ─────────────────────────────────────────────
df['CANTIDAD'] = pd.to_numeric(df['CANTIDAD'], errors='coerce')

print(' T2.2 — CANTIDAD convertida a numérico')
print(f'   Tipo                    : {df["CANTIDAD"].dtype}')
print(f'   Valores no convertibles : {df["CANTIDAD"].isna().sum()}')
print(f'   Mínimo                  : {df["CANTIDAD"].min():.0f}')
print(f'   Máximo                  : {df["CANTIDAD"].max():.0f}')
print(f'   Mediana                 : {df["CANTIDAD"].median():.0f}')

 T2.2 — CANTIDAD convertida a numérico
   Tipo                    : float64
   Valores no convertibles : 59
   Mínimo                  : 1
   Máximo                  : 304475
   Mediana                 : 20


In [119]:
# ── T2.3  Validar TIPO_SOLICITUD: exactamente 3 valores permitidos ─────────
# Según la Resolución 1478 de 2006, solo existen 3 tipos válidos
TIPOS_VALIDOS = {
    'PACIENTE ESPECIFICO',
    'MAS DE UN PACIENTE',
    'URGENCIA CLINICA'
}

valores_encontrados = set(df['TIPO_SOLICITUD'].dropna().unique())
valores_invalidos   = valores_encontrados - TIPOS_VALIDOS

print(' T2.3 — Validación TIPO_SOLICITUD')
print(f'   Valores encontrados  : {valores_encontrados}')
print(f'   Valores inválidos    : {valores_invalidos if valores_invalidos else "Ninguno "}')
print()
print('   Distribución:')
print(df['TIPO_SOLICITUD'].value_counts().to_string())

if valores_invalidos:
    raise ValueError(
        f'TIPO_SOLICITUD tiene valores no reconocidos: {valores_invalidos}. '
        'Verifica que T1.3 (quitar tildes) se ejecutó antes de esta celda.'
    )

 T2.3 — Validación TIPO_SOLICITUD
   Valores encontrados  : {'MAS DE UN PACIENTE', 'URGENCIA CLINICA', 'PACIENTE ESPECIFICO'}
   Valores inválidos    : Ninguno 

   Distribución:
TIPO_SOLICITUD
PACIENTE ESPECIFICO    5594
MAS DE UN PACIENTE     3467
URGENCIA CLINICA        956


---
## T3 — TRATAR VALORES FALTANTES Y ELIMINACION DE DATOS DUPLICADOS

Dos acciones en este paso:

- T3.1 — Nulos NaN real:* que significan ausencia de dato pero Python interpreta sin informacion.
- T3.2 — Eliminar columnas con >70% de datos faltantes:** columnas que no aportan información suficiente para el análisis. Antes de eliminarlas, extraemos cualquier dato útil que contengan. asi como valores duplicados

In [120]:
# ── T3.1  NULOS REALES POR COLUMNA ─────────────────────────────

import numpy as np

VALORES_NULOS = [
    'NO APLICA', 'NO REPORTADO', 'NO REPORTA',
    'N/A', 'NA', 'SIN INFORMACION', '-', '.', ''
]

print('Nulos disfrazados encontrados:')
total_disfrazados = 0
for col in df.select_dtypes(include='object').columns:
    n = df[col].isin(VALORES_NULOS).sum()
    if n > 0:
        print(f'   {col:<35}: {n:,} valores')
        total_disfrazados += n

# Conteo de nulos reales (NaN)
nulos_por_columna = df.isnull().sum()
print(' Nulos reales por columna:\n')
total_nulos = 0
for col, n in nulos_por_columna.items():
    if n > 0:
        porcentaje = (n / len(df)) * 100
        print(f'   {col:<35}: {n:,} ({porcentaje:.1f}%)')
        total_nulos += n

print(f'\n✔ Total de nulos reales en el dataset: {total_nulos:,}')

# 2. TRATAR VARIABLE NUMÉRICA: CANTIDAD

# Asegurar tipo numérico
df['CANTIDAD'] = pd.to_numeric(df['CANTIDAD'], errors='coerce')
# Contar nulos antes
nulos_antes = df['CANTIDAD'].isnull().sum()
# Calcular mediana
mediana = df['CANTIDAD'].median()
# Imputar con mediana
df['CANTIDAD'] = df['CANTIDAD'].fillna(mediana)
# Contar nulos después
nulos_despues = df['CANTIDAD'].isnull().sum()

print('\n IMPUTACIÓN EN CANTIDAD:')

print(f'   Nulos antes        : {nulos_antes:,}')
print(f'   Mediana usada      : {mediana}')
print(f'   Nulos después      : {nulos_despues:,}')
print(f'   Valores imputados  : {nulos_antes - nulos_despues:,}')

Nulos disfrazados encontrados:
   IUM                                : 5 valores
   PRINCIPIO_ACTIVO2                  : 9,176 valores
   CONCENTRACION2                     : 9,167 valores
   UNIDAD_MEDIDA2                     : 9,168 valores
   NOMBRE_COMERCIAL                   : 957 valores
   PRESENTACION_COMERCIAL             : 37 valores
   DIAGNOSTICO                        : 1,338 valores
   CODIGO_CIE10                       : 1,777 valores
 Nulos reales por columna:

   CONCENTRACION2                     : 19 (0.2%)
   UNIDAD_MEDIDA2                     : 21 (0.2%)
   CANTIDAD                           : 59 (0.6%)

✔ Total de nulos reales en el dataset: 99

 IMPUTACIÓN EN CANTIDAD:
   Nulos antes        : 59
   Mediana usada      : 20.0
   Nulos después      : 0
   Valores imputados  : 59


In [124]:
# ── T3.2  Eliminar columnas no funcionales ─────────────────────────────

COLUMNAS_NO_FUNCIONALES = [
    'PRINCIPIO_ACTIVO2',
    'CONCENTRACION2',
    'UNIDAD_MEDIDA2'
]

for col in COLUMNAS_NO_FUNCIONALES:
    pct = df[col].isnull().mean() * 100
    print(f'{col:<30}: {pct:.1f}% nulos')

# Crear variable derivada correctamente
df['ES_COMBINADO'] = (
    df['PRINCIPIO_ACTIVO2']
    .fillna('')
    .astype(str)
    .str.strip()
    .replace('NO APLICA', '')
    .ne('')
).astype(int)

n_combinados = df['ES_COMBINADO'].sum()

print(f'\nMedicamentos combinados: {n_combinados:,} ({n_combinados/len(df)*100:.1f}%)')

# Eliminar columnas
df = df.drop(columns=COLUMNAS_NO_FUNCIONALES)

# Reporte final
print(f'\nColumnas eliminadas: {len(COLUMNAS_NO_FUNCIONALES)}')
print(f'Dimensión actual: {df.shape[0]:,} filas × {df.shape[1]} columnas')


KeyError: 'PRINCIPIO_ACTIVO2'

In [123]:
# CONTEO Y ELIMINACIÓN DE DUPLICADOS
# Dimensión inicial
filas_antes, columnas_antes = df.shape
duplicados_encontrados = df.duplicated().sum()
df = df.drop_duplicates()

filas_despues, columnas_despues = df.shape
# Cantidad eliminada
duplicados_eliminados = filas_antes - filas_despues

print(' REPORTE DE DUPLICADOS\n')

print(f'   Filas antes del proceso     : {filas_antes:,}')
print(f'   Duplicados encontrados      : {duplicados_encontrados:,}')
print(f'   Duplicados eliminados       : {duplicados_eliminados:,}')
print(f'   Filas después de limpieza   : {filas_despues:,}')

 REPORTE DE DUPLICADOS

   Filas antes del proceso     : 9,413
   Duplicados encontrados      : 0
   Duplicados eliminados       : 0
   Filas después de limpieza   : 9,413


In [125]:
# ── Reporte de nulos en el dataset después de T3 ──────────────────────────
print('Estado de nulos después de T3:')
nulos_actuales = df.isna().sum()
pct_actuales   = (nulos_actuales / len(df) * 100).round(1)

reporte = pd.DataFrame({
    'Nulos': nulos_actuales,
    '% Faltante': pct_actuales
}).sort_values('Nulos', ascending=False)

print(reporte.to_string())

Estado de nulos después de T3:
                        Nulos  % Faltante
FECHA_AUTORIZACION          0         0.0
TIPO_SOLICITUD              0         0.0
IMPORTADOR                  0         0.0
IUM                         0         0.0
PRINCIPIO_ACTIVO            0         0.0
CONCENTRACION               0         0.0
UNIDAD_MEDIDA               0         0.0
FORMA_FARMACEUTICA          0         0.0
NOMBRE_COMERCIAL            0         0.0
CANTIDAD                    0         0.0
PRESENTACION_COMERCIAL      0         0.0
DIAGNOSTICO                 0         0.0
CODIGO_CIE10                0         0.0
ES_COMBINADO                0         0.0


---
## T4 — NORMALIZACIÓN DE TODOS LOS TIPOS DE DATO

La normalización va más allá del texto. Cada tipo de dato tiene su propia normalización:

| Tipo | ¿Qué se normaliza? |
|------|--------------------|
| **Texto** | Eliminar tildes, puntos de abreviatura, caracteres especiales |
| **Fechas** | Validar que estén dentro del rango 2018–2026 |
| **Números** | Validar que CANTIDAD sea positiva y no tenga valores imposibles |
| **Categorías** | Que TIPO_SOLICITUD y FORMA_FARMACEUTICA solo tengan valores coherentes |

In [126]:
# ── T4.1  Normalización profunda de texto ─────────────────────────────────
# Aplicar a columnas de texto clave: eliminar tildes y puntos de abreviatura
# Ejemplo: 'AUDIFARMA S.A.' → 'AUDIFARMA SA'

def normalizar_profundo(texto):
    """Elimina tildes, puntos de abreviatura y caracteres especiales."""
    if pd.isna(texto):
        return np.nan
    texto = str(texto)
    # Eliminar tildes (NFD descompone la letra+tilde, luego filtramos la tilde)
    texto = unicodedata.normalize('NFD', texto)
    texto = ''.join(c for c in texto if unicodedata.category(c) != 'Mn')
    # Quitar puntos de abreviatura: S.A. → SA, S.A.S. → SAS
    texto = re.sub(r'\.(?=[A-Z\s])', '', texto)
    # Limpiar espacios residuales
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

COLS_NORMALIZAR_TEXTO = [
    'TIPO_SOLICITUD',
    'IMPORTADOR',
    'PRINCIPIO_ACTIVO',
    'NOMBRE_COMERCIAL',
    'FORMA_FARMACEUTICA',
    'DIAGNOSTICO',
    'CODIGO_CIE10'
]

print(' T4.1 — Normalización profunda de texto:')
for col in COLS_NORMALIZAR_TEXTO:
    if col not in df.columns:
        continue
    unicos_antes  = df[col].nunique()
    df[col]       = df[col].apply(normalizar_profundo)
    unicos_despues = df[col].nunique()
    diff = unicos_antes - unicos_despues
    print(f'   {col:<30}: {unicos_antes} → {unicos_despues} valores únicos  (reducción: {diff})')

 T4.1 — Normalización profunda de texto:
   TIPO_SOLICITUD                : 3 → 3 valores únicos  (reducción: 0)
   IMPORTADOR                    : 302 → 302 valores únicos  (reducción: 0)
   PRINCIPIO_ACTIVO              : 587 → 587 valores únicos  (reducción: 0)
   NOMBRE_COMERCIAL              : 796 → 791 valores únicos  (reducción: 5)
   FORMA_FARMACEUTICA            : 74 → 74 valores únicos  (reducción: 0)
   DIAGNOSTICO                   : 672 → 672 valores únicos  (reducción: 0)
   CODIGO_CIE10                  : 561 → 561 valores únicos  (reducción: 0)


In [127]:
# ── T4.2  Normalización de fechas: validar rango 2018–2026 ────────────────
FECHA_MIN = pd.Timestamp('2018-01-01')
FECHA_MAX = pd.Timestamp('2026-12-31')

fuera_rango = df[
    (df['FECHA_AUTORIZACION'] < FECHA_MIN) |
    (df['FECHA_AUTORIZACION'] > FECHA_MAX)
]

print(' T4.2 — Validación de rango de fechas (2018–2026):')
print(f'   Fechas dentro del rango  : {len(df) - len(fuera_rango):,}')
print(f'   Fechas fuera del rango   : {len(fuera_rango)}')

if len(fuera_rango) > 0:
    print('     Registros con fecha fuera de rango:')
    print(fuera_rango[['FECHA_AUTORIZACION','IMPORTADOR','PRINCIPIO_ACTIVO']].to_string(index=False))
else:
    print('   Todas las fechas están dentro del período del proyecto ')

 T4.2 — Validación de rango de fechas (2018–2026):
   Fechas dentro del rango  : 9,413
   Fechas fuera del rango   : 0
   Todas las fechas están dentro del período del proyecto 


In [128]:
# ── T4.3  Normalización de números: validar que CANTIDAD sea positiva ──────
negativos = (df['CANTIDAD'] < 0).sum()
ceros     = (df['CANTIDAD'] == 0).sum()

print(' T4.3 — Validación de CANTIDAD:')
print(f'   Valores negativos (imposibles)  : {negativos}')
print(f'   Valores en cero (cuestionables) : {ceros}')
print(f'   Valores válidos (> 0)           : {(df["CANTIDAD"] > 0).sum():,}')

if negativos > 0:
    print('     Hay cantidades negativas → convertir a NaN')
    df.loc[df['CANTIDAD'] < 0, 'CANTIDAD'] = np.nan
else:
    print('   No hay cantidades negativas ')

# Crear categoría de tamaño de solicitud
def categorizar_cantidad(x):
    if pd.isna(x):   return np.nan
    elif x <= 10:    return '1-PEQUEÑA (1-10)'
    elif x <= 100:   return '2-MEDIANA (11-100)'
    elif x <= 1000:  return '3-GRANDE (101-1000)'
    else:            return '4-MUY GRANDE (>1000)'

df['CANTIDAD_CATEGORIA'] = df['CANTIDAD'].apply(categorizar_cantidad)

print('\n   Distribución por categoría de cantidad:')
print(df['CANTIDAD_CATEGORIA'].value_counts().sort_index().to_string())

 T4.3 — Validación de CANTIDAD:
   Valores negativos (imposibles)  : 0
   Valores en cero (cuestionables) : 0
   Valores válidos (> 0)           : 9,413
   No hay cantidades negativas 

   Distribución por categoría de cantidad:
CANTIDAD_CATEGORIA
1-PEQUEÑA (1-10)        3494
2-MEDIANA (11-100)      3204
3-GRANDE (101-1000)     1141
4-MUY GRANDE (>1000)    1574


---
## T5 — CREAR VARIABLES DERIVADAS

Las variables derivadas son columnas **calculadas a partir de los datos existentes**.  
No modifican el dato original, pero lo convierten en algo directamente utilizable para el análisis de cada sprint.

| Variable | Origen | Uso en el proyecto |
|---|---|---|
| `ANIO` | FECHA_AUTORIZACION | Análisis de tendencia anual |
| `MES` | FECHA_AUTORIZACION | Análisis de estacionalidad  |
| `TRIMESTRE` | FECHA_AUTORIZACION | Agrupaciones trimestrales |
| `AÑO_PARCIAL` | ANIO | Marcar 2026 como año incompleto |
| `CAPITULO_CIE10` | CODIGO_CIE10 | Agrupar diagnósticos por sistema del cuerpo |
| `ES_URGENCIA` | TIPO_SOLICITUD | Variable binaria para análisis estadístico |
| `ES_COMBINADO` | PRINCIPIO_ACTIVO2 | Ya creada en T3 |

In [129]:
# ── T5.1  Variables temporales ────────────────────────────────────────────
df['ANIO']       = df['FECHA_AUTORIZACION'].dt.year
df['MES']        = df['FECHA_AUTORIZACION'].dt.month
df['TRIMESTRE']  = df['FECHA_AUTORIZACION'].dt.quarter

# Marcar el año más reciente como parcial (datos incompletos)
anio_max = df['ANIO'].max()
df['ANIO_PARCIAL'] = (df['ANIO'] == anio_max).astype(int)

print(' T5.1 — Variables temporales creadas:')
print(f'   ANIO      : {df["ANIO"].min()} – {df["ANIO"].max()}')
print(f'   MES       : 1 – 12')
print(f'   TRIMESTRE : 1 – 4')
print(f'   ANIO_PARCIAL = 1 para el año {anio_max} ({df["ANIO_PARCIAL"].sum()} registros)')
print()
print('   Registros por año:')
print(df.groupby('ANIO').size().to_string())

 T5.1 — Variables temporales creadas:
   ANIO      : 2018 – 2026
   MES       : 1 – 12
   TRIMESTRE : 1 – 4
   ANIO_PARCIAL = 1 para el año 2026 (356 registros)

   Registros por año:
ANIO
2018    1025
2019     839
2020     692
2021     923
2022    1217
2023    1307
2024    1554
2025    1500
2026     356


In [132]:
# ── T5.2  Capítulo CIE-10 ────────────────────────────────────────────────
# La letra inicial del código CIE-10 identifica el capítulo (sistema del cuerpo)
# Ejemplo: E849 → E → Enfermedades endocrinas, nutricionales y metabólicas

CAPITULOS_CIE10 = {
    'A': 'Enfermedades infecciosas y parasitarias',
    'B': 'Enfermedades infecciosas y parasitarias',
    'C': 'Neoplasias malignas (oncológico)',
    'D': 'Neoplasias benignas / Enf. de la sangre',
    'E': 'Enf. endocrinas, nutricionales y metabólicas',
    'F': 'Trastornos mentales y del comportamiento',
    'G': 'Enfermedades del sistema nervioso',
    'H': 'Enfermedades del ojo y del oído',
    'I': 'Enfermedades del sistema circulatorio',
    'J': 'Enfermedades del sistema respiratorio',
    'K': 'Enfermedades del sistema digestivo',
    'L': 'Enfermedades de la piel',
    'M': 'Enfermedades del sistema musculoesquelético',
    'N': 'Enfermedades del sistema genitourinario',
    'O': 'Embarazo, parto y puerperio',
    'P': 'Afecciones del período perinatal',
    'Q': 'Malformaciones congénitas',
    'R': 'Síntomas y signos no clasificados',
    'S': 'Traumatismos y envenenamientos',
    'T': 'Traumatismos y envenenamientos',
    'Z': 'Factores que influyen en el estado de salud'
}

def extraer_capitulo(codigo):
    if pd.isna(codigo) or str(codigo).strip() == '':
        return np.nan
    letra = str(codigo).strip().upper()[0]
    return CAPITULOS_CIE10.get(letra, f'Otro (capítulo {letra})')

df['CAPITULO_CIE10'] = df['CODIGO_CIE10'].apply(extraer_capitulo)

print('T5.2 — CAPITULO_CIE10 creado')
print('\n   Distribución por capítulo:')
print(df['CAPITULO_CIE10'].value_counts().to_string())

T5.2 — CAPITULO_CIE10 creado

   Distribución por capítulo:
CAPITULO_CIE10
Enf. endocrinas, nutricionales y metabólicas    2050
Enfermedades del sistema genitourinario         1834
Enfermedades del sistema nervioso               1425
Neoplasias malignas (oncológico)                1342
Malformaciones congénitas                        836
Neoplasias benignas / Enf. de la sangre          587
Traumatismos y envenenamientos                   472
Enfermedades del sistema circulatorio            158
Enfermedades infecciosas y parasitarias          124
Enfermedades del ojo y del oído                  120
Factores que influyen en el estado de salud      102
Síntomas y signos no clasificados                 91
Enfermedades de la piel                           64
Enfermedades del sistema digestivo                54
Enfermedades del sistema musculoesquelético       44
Trastornos mentales y del comportamiento          40
Enfermedades del sistema respiratorio             23
Afecciones del período p

In [131]:
# ── T5.3  Indicador de urgencia clínica ──────────────────────────────────
# Variable binaria: 1 = urgencia clínica, 0 = los otros dos tipos
# Será clave para el análisis estadístico (chi-cuadrado) en el Sprint 4

df['ES_URGENCIA'] = (df['TIPO_SOLICITUD'] == 'URGENCIA CLINICA').astype(int)

print(' T5.3 — ES_URGENCIA creado')
print(f'   Urgencias clínicas : {df["ES_URGENCIA"].sum():,} ({df["ES_URGENCIA"].mean()*100:.1f}%)')
print(f'   Otros tipos        : {(df["ES_URGENCIA"]==0).sum():,} ({(df["ES_URGENCIA"]==0).mean()*100:.1f}%)')

 T5.3 — ES_URGENCIA creado
   Urgencias clínicas : 944 (10.0%)
   Otros tipos        : 8,469 (90.0%)


In [133]:
# ── Resumen de variables derivadas ───────────────────────────────────────
vars_derivadas = ['ANIO','MES','TRIMESTRE','ANIO_PARCIAL',
                  'CAPITULO_CIE10','ES_URGENCIA','ES_COMBINADO','CANTIDAD_CATEGORIA']

print('Variables derivadas disponibles en el dataset limpio:')
for v in vars_derivadas:
    print(f'   {v:<25}: {df[v].nunique()} valores únicos')

Variables derivadas disponibles en el dataset limpio:
   ANIO                     : 9 valores únicos
   MES                      : 12 valores únicos
   TRIMESTRE                : 4 valores únicos
   ANIO_PARCIAL             : 2 valores únicos
   CAPITULO_CIE10           : 22 valores únicos
   ES_URGENCIA              : 2 valores únicos
   ES_COMBINADO             : 2 valores únicos
   CANTIDAD_CATEGORIA       : 4 valores únicos


---
## T6 — VALIDACIÓN FINAL

Antes de exportar verificamos que el dataset cumple las condiciones mínimas de calidad.  
Este checklist forma parte del informe académico como evidencia de rigor metodológico.

In [134]:
# ── Comparación antes vs. después del ETL ─────────────────────────────────
print('COMPARACIÓN ANTES vs. DESPUÉS DEL ETL:')
print(f'   {'':30} {'ANTES':>10} {'DESPUÉS':>10}')
print(f"   {'-'*50}")
print(f"   {'Filas':30} {df_crudo.shape[0]:>10,} {df.shape[0]:>10,}")
print(f"   {'Columnas':30} {df_crudo.shape[1]:>10} {df.shape[1]:>10}")
print(f"   {'Columnas eliminadas':30} {'':>10} {df_crudo.shape[1]-df.shape[1]:>10}")
print(f"   {'Variables derivadas nuevas':30} {'':>10} {df.shape[1]-df_crudo.shape[1]+len(cols_eliminar):>10}")
print(f"   {'Nulos totales':30} {df_crudo.isna().sum().sum():>10,} {df.isna().sum().sum():>10,}")

COMPARACIÓN ANTES vs. DESPUÉS DEL ETL:
                                       ANTES    DESPUÉS
   --------------------------------------------------
   Filas                              10,017      9,413
   Columnas                               16         21
   Columnas eliminadas                               -5
   Variables derivadas nuevas                         5
   Nulos totales                          99          0


---
##  L — CARGA: Exportar el dataset limpio

Generamos dos archivos:
1. **`mvnd_limpio.csv`** — Dataset listo para usar en todos los notebooks de análisis
2. **`reporte_etl.xlsx`** — Documento de trazabilidad para el informe académico

In [135]:
# ── Exportar CSV limpio ───────────────────────────────────────────────────
# utf-8-sig garantiza que Excel abra correctamente las tildes y caracteres especiales
df.to_csv(RUTA_LIMPIO, index=False, encoding='utf-8-sig')

print(f' Dataset limpio exportado: {RUTA_LIMPIO}')
print(f'   Filas exportadas   : {len(df):,}')
print(f'   Columnas exportadas: {len(df.columns)}')
print(f'   Columnas: {list(df.columns)}')

 Dataset limpio exportado: ../datasets/mvnd_limpio.csv
   Filas exportadas   : 9,413
   Columnas exportadas: 21
   Columnas: ['FECHA_AUTORIZACION', 'TIPO_SOLICITUD', 'IMPORTADOR', 'IUM', 'PRINCIPIO_ACTIVO', 'CONCENTRACION', 'UNIDAD_MEDIDA', 'FORMA_FARMACEUTICA', 'NOMBRE_COMERCIAL', 'CANTIDAD', 'PRESENTACION_COMERCIAL', 'DIAGNOSTICO', 'CODIGO_CIE10', 'ES_COMBINADO', 'CANTIDAD_CATEGORIA', 'ANIO', 'MES', 'TRIMESTRE', 'ANIO_PARCIAL', 'CAPITULO_CIE10', 'ES_URGENCIA']


In [137]:
# ── Resumen ejecutivo final ───────────────────────────────────────────────
print('=' * 62)
print('  RESUMEN EJECUTIVO DEL ETL')
print('=' * 62)
print(f"""
 ENTRADA
   {df_crudo.shape[0]:,} filas  ×  {df_crudo.shape[1]} columnas
   Archivo: MEDICAMENTOS_VITALES_NO_DISPONIBLES_20260425.xlsx

 TRANSFORMACIONES
   T1 → Encabezados renombrados + todos los valores de texto
        estandarizados (mayúsculas, sin espacios extra)
   T2 → Fecha sin hora, cantidad numérica, categorías validadas
   T3 → {total_disfrazados:,} nulos disfrazados convertidos a NaN
        {len(cols_eliminar)} columnas con >{int(UMBRAL_NULOS*100)}% vacías eliminadas
   T4 → Texto: tildes y abreviaturas eliminadas
        Fechas: rango 2018–2026 validado
        Números: negativos corregidos, categorías creadas
        Categorías: formas farmacéuticas consolidadas
   T5 → 7 variables derivadas nuevas creadas

SALIDA
   {df.shape[0]:,} filas  ×  {df.shape[1]} columnas
   Archivo: mvnd_limpio.csv

DECISIONES CLAVE
   • NO se eliminó ningún registro original
   • Columnas eliminadas: {cols_eliminar}
   • ES_COMBINADO creado antes de eliminar las columnas vacías
   • 2026 marcado como año parcial (AÑO_PARCIAL = 1)
   • Los {df['DIAGNOSTICO'].isna().sum():,} registros sin diagnóstico se conservan con NaN

SIGUIENTE PASO
   Notebook 03 — Sprint 1: Análisis de Medicamentos
   Cargar: mvnd_limpio.csv
""")
print('=' * 62)

  RESUMEN EJECUTIVO DEL ETL

 ENTRADA
   10,017 filas  ×  16 columnas
   Archivo: MEDICAMENTOS_VITALES_NO_DISPONIBLES_20260425.xlsx

 TRANSFORMACIONES
   T1 → Encabezados renombrados + todos los valores de texto
        estandarizados (mayúsculas, sin espacios extra)
   T2 → Fecha sin hora, cantidad numérica, categorías validadas
   T3 → 31,625 nulos disfrazados convertidos a NaN
        0 columnas con >90% vacías eliminadas
   T4 → Texto: tildes y abreviaturas eliminadas
        Fechas: rango 2018–2026 validado
        Números: negativos corregidos, categorías creadas
        Categorías: formas farmacéuticas consolidadas
   T5 → 7 variables derivadas nuevas creadas

SALIDA
   9,413 filas  ×  21 columnas
   Archivo: mvnd_limpio.csv

DECISIONES CLAVE
   • NO se eliminó ningún registro original
   • Columnas eliminadas: []
   • ES_COMBINADO creado antes de eliminar las columnas vacías
   • 2026 marcado como año parcial (AÑO_PARCIAL = 1)
   • Los 0 registros sin diagnóstico se conservan c